# 1강 실습: 이미지는 숫자다

**사용법**: 위에서부터 회색 코드 칸을 클릭하고 `Shift + Enter` 를 누르세요. 결과가 칸 아래에 나옵니다.

오늘 할 것
1. 사진을 불러와서 숫자로 들여다보기
2. 흑백 vs 컬러, OpenCV의 BGR 함정
3. 밝기·대비 바꾸기, 히스토그램
4. **미니 프로젝트**: 금속판 사진에서 불량(긁힘·이물) 자동으로 찾기

> 그래프 제목은 영어로 씁니다. Colab에는 한글 글꼴이 기본으로 없어서 한글이 네모(□)로 깨지기 때문입니다.

## 0. 준비: 라이브러리 불러오기
도구 상자를 여는 단계입니다. Colab에는 이미 설치되어 있습니다.

In [ ]:
import numpy as np                  # 숫자 표(행렬) 다루기
import cv2                          # OpenCV: 이미지 처리
import matplotlib.pyplot as plt     # 그림 그리기
from skimage import data            # 연습용 샘플 사진 모음

print("NumPy  버전:", np.__version__)
print("OpenCV 버전:", cv2.__version__)
print("준비 완료!")

## 1. 흑백 사진 불러오기

`data.camera()` 는 연습용 흑백 사진입니다. 사진을 불러온 뒤 **크기(shape)**, **형식(dtype)**, **가장 어두운/밝은 값** 을 확인합니다.

In [ ]:
img = data.camera()

print("크기(세로, 가로):", img.shape)
print("형식:", img.dtype)                 # uint8 = 0~255 정수
print("가장 어두운 값:", img.min(), "/ 가장 밝은 값:", img.max())
print("전체 픽셀 개수:", img.size)

plt.imshow(img, cmap="gray")
plt.title("Grayscale image")
plt.axis("off")
plt.show()

## 2. 확대해서 숫자 직접 보기

사진의 일부(10×10 픽셀)만 잘라서 **실제 숫자**를 출력해 봅니다. 사진이 정말 숫자 표인지 눈으로 확인하는 단계입니다.

`img[행 시작:행 끝, 열 시작:열 끝]` 은 표의 일부를 잘라내는 문법입니다.

In [ ]:
patch = img[100:110, 200:210]     # 세로 100~109, 가로 200~209 영역
print(patch)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(patch, cmap="gray", vmin=0, vmax=255)
for y in range(patch.shape[0]):
    for x in range(patch.shape[1]):
        v = patch[y, x]
        ax.text(x, y, v, ha="center", va="center", fontsize=8,
                color="red" if v > 128 else "yellow")
ax.set_title("10x10 patch: each square is one number")
ax.axis("off")
plt.show()

✏️ **해 보기**: 위 칸에서 `img[100:110, 200:210]` 의 숫자를 바꿔서 다른 위치를 잘라 보세요. 밝은 하늘 부분(예: `img[10:20, 10:20]`)은 숫자가 큽니다.

## 3. 컬러 사진 = 숫자 표 3장

컬러 사진은 `(세로, 가로, 3)` 모양입니다. 마지막 3이 **R, G, B** 세 장의 표입니다.

In [ ]:
color = data.coffee()        # 커피잔 컬러 사진 (RGB 순서)
print("크기:", color.shape)
print("왼쪽 위 픽셀의 (R, G, B):", color[0, 0])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(color);                    axes[0].set_title("Color (RGB)")
axes[1].imshow(color[:, :, 0], cmap="Reds");   axes[1].set_title("R channel")
axes[2].imshow(color[:, :, 1], cmap="Greens"); axes[2].set_title("G channel")
axes[3].imshow(color[:, :, 2], cmap="Blues");  axes[3].set_title("B channel")
for a in axes: a.axis("off")
plt.show()

### OpenCV의 BGR 함정

OpenCV(`cv2`)는 색 순서를 **B, G, R** 로 씁니다. 공장 카메라 사진을 `cv2.imread()` 로 읽으면 BGR 순서가 됩니다. 아래에서 순서를 뒤집으면 색이 어떻게 이상해지는지 보세요.

In [ ]:
bgr = cv2.cvtColor(color, cv2.COLOR_RGB2BGR)   # OpenCV로 읽었을 때와 같은 상태로 만들기

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(bgr);  axes[0].set_title("WRONG: BGR shown as RGB")
axes[1].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)); axes[1].set_title("RIGHT: converted to RGB")
for a in axes: a.axis("off")
plt.show()

gray = cv2.cvtColor(color, cv2.COLOR_RGB2GRAY)  # 컬러 → 흑백 변환
print("흑백으로 바꾼 크기:", gray.shape, "(채널 3 → 1)")

## 4. 밝기·대비 바꾸기

- 밝기: 모든 숫자에 **더하기**
- 대비: 모든 숫자에 **곱하기**
- 255를 넘거나 0보다 작아지지 않게 `np.clip` 으로 잘라 줍니다.

> 왜 `astype(np.int16)` 을 할까요? `uint8` 은 255를 넘으면 0부터 다시 시작(250+10=4)하는 성질이 있어서, 계산할 때만 더 큰 형식으로 바꿉니다.

In [ ]:
def adjust(image, brightness=0, contrast=1.0):
    out = image.astype(np.int16) * contrast + brightness
    return np.clip(out, 0, 255).astype(np.uint8)

brightness = 60     # ✏️ 숫자를 바꿔 보세요 (-100 ~ 100)
contrast   = 1.5    # ✏️ 숫자를 바꿔 보세요 (0.5 ~ 2.0)

versions = {
    "Original": img,
    f"Brightness {brightness:+d}": adjust(img, brightness=brightness),
    f"Contrast x{contrast}": adjust(img, contrast=contrast),
}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for a, (name, im) in zip(axes, versions.items()):
    a.imshow(im, cmap="gray", vmin=0, vmax=255); a.set_title(name); a.axis("off")
plt.show()

## 5. 히스토그램: 밝기 분포 그래프

가로축 = 밝기(0~255), 세로축 = 그 밝기인 픽셀 개수. 밝게 만든 사진은 그래프가 오른쪽으로 이동합니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for a, (name, im) in zip(axes, versions.items()):
    a.hist(im.ravel(), bins=256, range=(0, 256), color="gray")
    a.set_title(name); a.set_xlabel("pixel value (0=black, 255=white)")
plt.show()

---
# 🏭 미니 프로젝트: 금속판 불량 자동 찾기

실제 공장 사진 대신, 컴퓨터로 **가짜 금속판 사진**을 만들어 씁니다.
- 양품: 표면 무늬만 있는 깨끗한 금속판
- 불량: 같은 금속판에 **긁힘(밝은 선)** 과 **이물(검은 점)** 을 추가

그리고 1강 이론 6번의 방법(정상 사진과 빼기)으로 불량을 찾아 **빨간 박스**로 표시합니다.

In [ ]:
rng = np.random.default_rng(seed=0)   # seed를 고정하면 매번 같은 결과가 나옵니다

def make_metal_plate(h=300, w=400):
    """가짜 금속판: 회색 바탕 + 가로 방향 결무늬 + 약간의 노이즈"""
    base = np.full((h, w), 150, dtype=np.float32)
    streaks = rng.normal(0, 6, size=(h, 1))          # 가로 결무늬
    base += streaks
    return base

plate = make_metal_plate()

def take_photo(plate):
    """카메라로 찍을 때마다 생기는 작은 노이즈를 더해 사진 1장을 만듦"""
    photo = plate + rng.normal(0, 3, size=plate.shape)
    return np.clip(photo, 0, 255).astype(np.uint8)

good = take_photo(plate)        # 기준이 될 양품 사진
test_good = take_photo(plate)   # 검사할 사진 ①: 양품

test_bad = take_photo(plate)    # 검사할 사진 ②: 불량 만들기
cv2.line(test_bad, (60, 220), (180, 180), color=230, thickness=2)   # 긁힘 (밝은 선)
cv2.circle(test_bad, (300, 90), radius=7, color=40, thickness=-1)  # 이물 (검은 점)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for a, (name, im) in zip(axes, [("Reference (good)", good), ("Test 1", test_good), ("Test 2", test_bad)]):
    a.imshow(im, cmap="gray", vmin=0, vmax=255); a.set_title(name); a.axis("off")
plt.show()

### 단계 ① 빼기: 기준 사진과 얼마나 다른가?

`cv2.absdiff` 는 두 사진의 픽셀끼리 빼서 **절댓값**(차이의 크기)을 구합니다. 같은 곳은 0에 가깝고(검정), 다른 곳만 밝게 나옵니다.

In [ ]:
diff_good = cv2.absdiff(good, test_good)
diff_bad  = cv2.absdiff(good, test_bad)

print("양품 사진 차이의 최댓값:", diff_good.max())
print("불량 사진 차이의 최댓값:", diff_bad.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(diff_good, cmap="hot", vmin=0, vmax=120); axes[0].set_title("Difference: Test 1")
axes[1].imshow(diff_bad,  cmap="hot", vmin=0, vmax=120); axes[1].set_title("Difference: Test 2")
for a in axes: a.axis("off")
plt.show()

### 단계 ② 기준선(임계값) 넘는 픽셀만 남기기

차이가 `THRESHOLD` 보다 큰 픽셀 = 흰색(255), 나머지 = 검정(0). 이런 흑백 결과를 **마스크** 라고 부릅니다.

✏️ `THRESHOLD` 를 5로 낮추면? 80으로 높이면? 직접 바꿔 보세요. 너무 낮으면 노이즈까지 불량으로 잡고(과검), 너무 높으면 불량을 놓칩니다(미검). **이 균형 잡기가 실제 검사 업무의 핵심**입니다.

In [ ]:
THRESHOLD = 30   # ✏️ 바꿔 보세요

_, mask_good = cv2.threshold(diff_good, THRESHOLD, 255, cv2.THRESH_BINARY)
_, mask_bad  = cv2.threshold(diff_bad,  THRESHOLD, 255, cv2.THRESH_BINARY)

print("양품 사진: 기준 넘는 픽셀 수 =", int((mask_good > 0).sum()))
print("불량 사진: 기준 넘는 픽셀 수 =", int((mask_bad > 0).sum()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(mask_good, cmap="gray"); axes[0].set_title("Mask: Test 1")
axes[1].imshow(mask_bad,  cmap="gray"); axes[1].set_title("Mask: Test 2")
for a in axes: a.axis("off")
plt.show()

### 단계 ③ 불량 위치에 박스 그리고 OK / NG 판정

마스크에서 흰 덩어리들을 찾아(`findContours`) 각 덩어리를 감싸는 빨간 박스를 그립니다. 덩어리가 하나라도 있으면 **NG**.

`MIN_AREA` 는 "이것보다 작은 점은 노이즈로 보고 무시"하는 기준입니다.

In [ ]:
MIN_AREA = 5   # ✏️ 픽셀 수 기준

def inspect(photo, reference, threshold=THRESHOLD, min_area=MIN_AREA):
    diff = cv2.absdiff(reference, photo)
    _, mask = cv2.threshold(diff, threshold, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    defects = [c for c in contours if cv2.contourArea(c) >= min_area or len(c) >= min_area]

    result = cv2.cvtColor(photo, cv2.COLOR_GRAY2RGB)   # 빨간 박스를 그리려고 컬러로 변환
    for c in defects:
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(result, (x - 4, y - 4), (x + w + 4, y + h + 4), (255, 0, 0), 2)
    verdict = "NG" if defects else "OK"
    return verdict, len(defects), result

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for a, (name, photo) in zip(axes, [("Test 1", test_good), ("Test 2", test_bad)]):
    verdict, n, result = inspect(photo, good)
    print(f"{name}: 판정 = {verdict}, 불량 개수 = {n}")
    a.imshow(result); a.set_title(f"{name}: {verdict} ({n} defects)"); a.axis("off")
plt.show()

### 단계 ④ 이 방법의 약점 직접 확인하기

같은 양품인데 **조명이 조금 밝아지면**(밝기 +40) 어떻게 될까요? 양품인데 NG가 나오는지 확인해 보세요.
이 문제를 해결하는 방법(전처리, 정규화)을 2강에서 배웁니다.

In [ ]:
brighter_good = adjust(test_good, brightness=40)    # 조명이 밝아진 양품
verdict, n, result = inspect(brighter_good, good)
print(f"조명이 밝아진 양품: 판정 = {verdict}  ← 양품인데 이렇게 나오면 '과검(오탐)'")

plt.figure(figsize=(7, 5))
plt.imshow(result); plt.title(f"Brighter good plate: {verdict}"); plt.axis("off")
plt.show()

---
## ✅ 1강 정리

- 이미지 = 숫자 표. 흑백은 1장, 컬러는 3장(RGB, OpenCV는 BGR).
- 밝기 = 더하기, 대비 = 곱하기, 히스토그램 = 밝기 분포.
- 가장 단순한 검사 = **정상 사진과 빼기 → 기준선 → 덩어리 찾기 → OK/NG**.
- 하지만 조명·위치가 바뀌면 쉽게 틀린다 → 그래서 전처리(2강)와 AI(4강~)가 필요하다.

## 📝 연습문제 (답은 Claude에게 물어보세요)

1. `data.camera()` 사진의 **평균 밝기**를 구해 보세요. (힌트: `img.mean()`)
2. 미니 프로젝트에서 긁힘을 하나 더 그려 넣고(`cv2.line`), 불량 개수가 3개로 나오는지 확인해 보세요.
3. `THRESHOLD` 를 바꿔 가며 "양품은 OK, 불량은 NG"가 둘 다 맞는 범위를 찾아보세요.
4. 생각해 보기: 우리 회사 검사 사진에서 **조명이나 위치가 바뀌는 상황**은 언제 생길까요?

## (선택) 내 사진 올려서 숫자로 보기

> ⚠️ **회사 사진은 올리지 마세요.** Colab에 올리면 외부(구글) 서버로 전송됩니다. 휴대폰으로 찍은 개인 사진으로 연습하세요.

아래 칸을 실행하면 파일 선택 버튼이 나옵니다. (Colab에서만 동작합니다.)

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        arr = cv2.imdecode(np.frombuffer(content, np.uint8), cv2.IMREAD_COLOR)  # BGR로 읽힘
        rgb = cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)
        print(name, "크기:", rgb.shape)
        plt.imshow(rgb); plt.axis("off"); plt.show()
except ImportError:
    print("이 칸은 Google Colab에서 실행할 때만 동작합니다.")